# NLI Combined Datasets: Gemma-2-2b-it (google/gemma-2-2b-it)

Tests **4 different dataset combinations** by concatenating training/test splits:

1. **SNLI + MNLI** (snli_tr_1_1 + multinli_tr_1_1)
2. **SNLI + TrGLUE** (snli_tr_1_1 + trglue_mnli)
3. **MNLI + TrGLUE** (multinli_tr_1_1 + trglue_mnli)
4. **SNLI + MNLI + TrGLUE** (all three datasets)

Loads [yilmazzey/sdp2-nli](https://huggingface.co/datasets/yilmazzey/sdp2-nli) and runs **test-only** evaluation with this model.

2B generative LLM (Gemma-2 series, instruct-tuned). English-only but multilingual pretraining. Zero-shot prompted NLI (no fine-tuning). Expected ~50-60% on reasoning/NLI benchmarks. Outputs parsed to 0=entailment, 1=neutral, 2=contradiction.

**Metrics:** Accuracy, macro F1, per-class F1, confusion matrix (CSV + plot) for each combination.

In [ ]:
# Colab: uncomment and run once to install/upgrade (Runtime -> Change runtime type -> GPU)
!pip install -q -U transformers datasets accelerate scikit-learn tqdm huggingface_hub[hf_transfer]

import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from tqdm import tqdm
from transformers import pipeline, AutoTokenizer

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOT = True
except ImportError:
    HAS_PLOT = False

LABEL_NAMES = ["entailment", "neutral", "contradiction"]

# Enable faster downloads
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

if torch.backends.mps.is_available():
    print("Apple Silicon MPS available; using for acceleration.")
elif torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU/MPS; running on CPU (2B still feasible but slower).")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
REPO_ID = "yilmazzey/sdp2-nli"
CONFIGS = ["snli_tr_1_1", "multinli_tr_1_1", "trglue_mnli"]
MODEL_ID = "google/gemma-2-2b-it"
NUM_LABELS = 3  # entailment, neutral, contradiction
RESULTS_DIR = "results_combined"
# Lower to 4-8 if memory low (2B lightweight). If CPU only, feasible.
BATCH_SIZE = 8  # Lowered for M4 36GB RAM; adjust if needed

# Define which splits to use for each config
EVAL_SPLITS = {
    "snli_tr_1_1": "test",
    "multinli_tr_1_1": ["validation_matched", "validation_mismatched"],
    "trglue_mnli": ["test_matched", "test_mismatched"],
}

In [ ]:
# Load all three dataset configs
datasets = {}
for cfg in CONFIGS:
    print(f"Loading {REPO_ID} :: {cfg} ...")
    datasets[cfg] = load_dataset(REPO_ID, cfg)
    print("  splits:", list(datasets[cfg].keys()))

In [ ]:
print("Loading Gemma-2-2b-it (text-generation pipeline)...")

# Load with MPS for M4 acceleration and bfloat16 for memory efficiency
device = "mps" if torch.backends.mps.is_available() else 0 if torch.cuda.is_available() else -1
generator = pipeline(
    "text-generation",
    model=MODEL_ID,
    device=device,
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if hasattr(tokenizer, "pad_token") and tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully.")

In [ ]:
def nli_prompt(premise, hypothesis):
    instruction = "You are a helpful assistant for natural language inference. Classify the relationship between premise and hypothesis as entailment, neutral, or contradiction. Respond with exactly one word: entailment, neutral, or contradiction."
    user_text = f"{instruction}\n\nPremise: {premise}\nHypothesis: {hypothesis}\nLabel:"
    return [{"role": "user", "content": user_text}]

def parse_generated_label(gen_text, prompt_text):
    continuation = gen_text.replace(prompt_text, "").strip().lower()
    if not continuation:
        return 1  # neutral fallback
    first_word = continuation.split()[0].rstrip('.')
    label_map = {"entailment": 0, "neutral": 1, "contradiction": 2}
    return label_map.get(first_word, 1)  # Default to neutral if unknown

def run_prompted_inference(ds):
    premises = list(ds["premise"])
    hypotheses = list(ds["hypothesis"])
    labels = list(ds["label"])
    n = len(ds)
    y_pred = []
    all_generations = []  # Collect all for full debug
    
    for start in tqdm(range(0, n, BATCH_SIZE), desc="Inference"):
        batch_premises = premises[start : start + BATCH_SIZE]
        batch_hypotheses = hypotheses[start : start + BATCH_SIZE]
        batch_prompts = [nli_prompt(p, h) for p, h in zip(batch_premises, batch_hypotheses)]
        
        formatted_prompts = tokenizer.apply_chat_template(batch_prompts, tokenize=False, add_generation_prompt=True)
        
        out = generator(
            formatted_prompts,
            max_new_tokens=5,  # Strict limit
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            max_length=None,  # Silence warning
        )
        
        all_generations.extend(out)  # Save all
        
        for i, gen in enumerate(out):
            gen_text = gen[0]["generated_text"]
            y_pred.append(parse_generated_label(gen_text, formatted_prompts[i]))
    
    y_true = np.array(labels, dtype=np.int64)
    y_pred = np.array(y_pred, dtype=np.int64)
    
    # Log first 5 generations for debug
    for i in range(min(5, n)):
        print(f"Debug Sample {i}: Generated: {all_generations[i][0]['generated_text']}, Parsed Label: {y_pred[i]}")
    
    print("True label dist:", dict(Counter(y_true)))
    print("Pred label dist:", dict(Counter(y_pred)))
    
    return y_true, y_pred

In [ ]:
def compute_metrics(y_true, y_pred):
    acc = float(accuracy_score(y_true, y_pred))
    f1_macro = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    f1_per_class = {LABEL_NAMES[i]: float(f1_per_class[i]) for i in range(NUM_LABELS)}
    cm = confusion_matrix(y_true, y_pred)
    out = {"accuracy": acc, "f1_macro": f1_macro, "f1_per_class": f1_per_class}
    return out, cm


def save_confusion_plot(cm, path):
    if not HAS_PLOT:
        return
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.tight_layout()
    plt.savefig(path)
    plt.close()

In [ ]:
# Helper function to get and concatenate datasets
def get_combined_dataset(config_names):
    """Concatenate test splits from multiple configs"""
    combined_parts = []
    
    for cfg in config_names:
        ds_dict = datasets[cfg]
        splits_to_use = EVAL_SPLITS[cfg]
        
        if isinstance(splits_to_use, str):
            splits_to_use = [splits_to_use]
        
        for split_name in splits_to_use:
            if split_name in ds_dict:
                combined_parts.append(ds_dict[split_name])
                print(f"  Added {cfg}/{split_name}: {len(ds_dict[split_name])} examples")
    
    if not combined_parts:
        return None
    
    combined = concatenate_datasets(combined_parts)
    print(f"  Total combined: {len(combined)} examples\n")
    return combined

In [ ]:
# Define the 4 dataset combinations
COMBINATIONS = {
    "snli+mnli": ["snli_tr_1_1", "multinli_tr_1_1"],
    "snli+trglue": ["snli_tr_1_1", "trglue_mnli"],
    "mnli+trglue": ["multinli_tr_1_1", "trglue_mnli"],
    "snli+mnli+trglue": ["snli_tr_1_1", "multinli_tr_1_1", "trglue_mnli"],
}

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
all_metrics = {}

for combo_name, config_list in COMBINATIONS.items():
    print(f"\n{'='*60}")
    print(f"Processing combination: {combo_name}")
    print(f"Configs: {config_list}")
    print(f"{'='*60}")
    
    # Get combined dataset
    combined_ds = get_combined_dataset(config_list)
    
    if combined_ds is None:
        print(f"  Skip {combo_name} (no valid splits)")
        continue
    
    # Run inference
    print(f"Evaluating {combo_name} ...")
    y_true, y_pred = run_prompted_inference(combined_ds)
    metrics, cm = compute_metrics(y_true, y_pred)
    all_metrics[combo_name] = metrics

    cm_path = Path(RESULTS_DIR) / f"confusion_{combo_name}.csv"
    np.savetxt(cm_path, cm, fmt="%d", delimiter=",")
    save_confusion_plot(cm, Path(RESULTS_DIR) / f"confusion_{combo_name}.png")

    print(f"  accuracy={metrics['accuracy']:.4f}, f1_macro={metrics['f1_macro']:.4f}")

with open(Path(RESULTS_DIR) / "metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved {RESULTS_DIR}/metrics.json")

In [ ]:
# Summary: all combinations
for combo_name, m in all_metrics.items():
    print(f"{combo_name}: acc={m['accuracy']:.4f}, F1_macro={m['f1_macro']:.4f}, F1_per_class={m['f1_per_class']}")